In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
from pyspark.sql.functions import col, explode_outer, when, size, current_timestamp

df_bronze = (
    spark.read
    .option("multiline", "true")
    .json("/mnt/raw/seoul_api/*.json")
)

# 날씨 — 보행 환경 분석에서 핵심 피처들 거의 다 여기 있음
df_weather = (
    df_bronze.select(
        col("CITYDATA.AREA_NM"),
        explode_outer(col("CITYDATA.WEATHER_STTS")).alias("weather")
    )
    .select(
        col("AREA_NM"),
        col("weather.TEMP"),
        col("weather.SENSIBLE_TEMP"),
        col("weather.HUMIDITY"),
        col("weather.WIND_SPD"),
        col("weather.WIND_DIRCT"),
        col("weather.PRECIPITATION"),
        col("weather.PRECPT_TYPE"),
        col("weather.PCP_MSG"),
        col("weather.UV_INDEX"),
        col("weather.UV_INDEX_LVL"),
        col("weather.PM10"),
        col("weather.PM10_INDEX"),
        col("weather.PM25"),
        col("weather.PM25_INDEX"),
        col("weather.AIR_IDX"),
        col("weather.AIR_MSG"),
        col("weather.WEATHER_TIME"),
    )
)
df_weather = df_weather.filter(col("AREA_NM").isNotNull())

# 혼잡도 — FCST_PPLTN(예측 혼잡) 포함, Gold에선 안 쓰지만 나중을 위해 남겨둠
df_congest = (
    df_bronze.select(
        col("CITYDATA.AREA_NM"),
        explode_outer(col("CITYDATA.LIVE_PPLTN_STTS")).alias("ppltn")
    )
    .select(
        col("AREA_NM"),
        col("ppltn.AREA_CONGEST_LVL"),
        col("ppltn.AREA_CONGEST_MSG"),
        col("ppltn.AREA_PPLTN_MIN"),
        col("ppltn.AREA_PPLTN_MAX"),
        col("ppltn.PPLTN_TIME"),
        col("ppltn.FCST_PPLTN"),
    )
)
df_congest = df_congest.filter(col("AREA_NM").isNotNull())

# 도로 — AVG_ROAD_DATA가 구역 대표값, 개별 링크는 나중에 지도 시각화용으로 쓸 수 있음
df_road = (
    df_bronze.select(
        col("CITYDATA.AREA_NM"),
        col("CITYDATA.ROAD_TRAFFIC_STTS.AVG_ROAD_DATA.ROAD_TRAFFIC_SPD"),
        col("CITYDATA.ROAD_TRAFFIC_STTS.AVG_ROAD_DATA.ROAD_TRAFFIC_IDX"),
        col("CITYDATA.ROAD_TRAFFIC_STTS.AVG_ROAD_DATA.ROAD_TRAFFIC_TIME"),
        explode_outer(col("CITYDATA.ROAD_TRAFFIC_STTS.ROAD_TRAFFIC_STTS")).alias("road"),
    )
    .select(
        col("AREA_NM"),
        col("ROAD_TRAFFIC_SPD"),
        col("ROAD_TRAFFIC_IDX"),
        col("ROAD_TRAFFIC_TIME"),
        col("road.LINK_ID"),
        col("road.ROAD_NM"),
        col("road.START_ND_XY"),
        col("road.END_ND_XY"),
        col("road.DIST"),
        col("road.SPD"),
        col("road.IDX"),
    )
)
df_road = df_road.filter(col("AREA_NM").isNotNull())

# 사고/행사/재난문자 — 있다/없다 수준으로만 쓸 거라 건수로 단순화
# EVENT_YN은 행사 여러 개여도 1로 처리 (행사 종류는 필요 없음)
df_event_accident = (
    df_bronze
    .filter(col("CITYDATA.AREA_NM").isNotNull())
    .select(
        col("CITYDATA.AREA_NM").alias("AREA_NM"),
        when(
            col("CITYDATA.ACDNT_CNTRL_STTS").isNull() |
            (size(col("CITYDATA.ACDNT_CNTRL_STTS")) == 0), 0
        ).otherwise(size(col("CITYDATA.ACDNT_CNTRL_STTS"))).alias("ACDNT_CNT"),
        when(
            col("CITYDATA.EVENT_STTS").isNull() |
            (size(col("CITYDATA.EVENT_STTS")) == 0), 0
        ).otherwise(1).alias("EVENT_YN"),
        when(
            col("CITYDATA.LIVE_DST_MESSAGE").isNull() |
            (size(col("CITYDATA.LIVE_DST_MESSAGE")) == 0), 0
        ).otherwise(size(col("CITYDATA.LIVE_DST_MESSAGE"))).alias("DST_MSG_CNT"),
        current_timestamp().alias("collected_at")
    )
)

df_weather.write.mode("overwrite").format("delta").saveAsTable("silver.weather")
df_congest.write.mode("overwrite").format("delta").saveAsTable("silver.congest")
df_road.write.mode("overwrite").format("delta").saveAsTable("silver.road")
df_event_accident.write.mode("overwrite").format("delta").saveAsTable("silver.event_accident")

print("✅ Silver 저장 완료")

✅ Silver 저장 완료! 🐾


In [0]:
spark.sql("SELECT * FROM silver.event_accident LIMIT 5").display()

AREA_NM,ACDNT_CNT,EVENT_YN,DST_MSG_CNT,collected_at
강남역,0,0,0,2026-04-09T07:25:46.834Z
가로수길,0,0,0,2026-04-09T07:25:46.834Z
강남 MICE 관광특구,0,0,0,2026-04-09T07:25:46.834Z
강남역,0,0,0,2026-04-09T07:25:46.834Z
강남역,0,0,0,2026-04-09T07:25:46.834Z
